# King County, Washington Housing Price Prediction and Grade Classification Using Feature Engineering, Feature Selection, and Hyperparameter Optimization


## 1.0 Business understanding

### 1.1 Situation analysis

Accurate house price forecasts are vital for buyers and sellers, real estate agents and mortgage lenders, home builders, and policymakers. 

Machine learning models can learn patterns from data, handle nonlinear relationships, and improve the accuracy of house price prediction. 

House price prediction to guide investment and policy decision-making is a primary area of research in data science.

### 1.2 Research question

**Research Question #1:**
Can housing price prediction be improved through feature engineering and selection, and by applying hyperparameter optimization to regression models?

**Research Question #2:**
Can the classification of housing grades be improved by applying feature engineering and selection to existing housing data, and hyperparameter optimization to classification models?

## 2.0 Data understanding

### 2.1 Preliminary data analysis

#### 2.1.1 Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib

In [ ]:
from scipy import stats
from catboost import CatBoostRegressor, CatBoostClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_regression, RFE, f_classif
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error, mean_squared_error
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve, auc
from sklearn.preprocessing import label_binarize, PowerTransformer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler, label_binarize, OneHotEncoder 
from sklearn.preprocessing import PowerTransformer, OneHotEncoder 

#### 2.1.2 Suppress warnings

In [ ]:
warnings.filterwarnings('ignore')

#### 2.1.3 Data ingestion

In [ ]:
df = pd.read_csv("kc_house_data.csv")

In [ ]:
df.shape

In [ ]:
df.head()

#### 2.1.4 Data description

* 1 - ida: notation for a house
* 2 - date: Date house was sold
* 3 - price: Price is prediction target
* 4 - bedrooms: Number of Bedrooms/House
* 5 - bathrooms: Number of bathrooms/House
* 6 - sqft_living: square footage of the home
* 7 - sqft_lot: square footage of the lot
* 8 - floors: Total floors (levels) in house
* 9 - waterfront: House which has a view to a waterfront
* 10 - view: Has been viewed
* 11 - condition: How good the condition is ( Overall )
* 12 - grade: overall grade given to the housing unit, based on King County grading system
* 13 - sqft_abovesquare: footage of house apart from basement
* 14 - sqft_basement: square footage of the basement
* 15 - yr_built: Built Year
* 16 - yr_renovated: Year when house was renovated
* 17 - zipcode: zip
* 18 - lat: Latitude coordinate
* 19 - long: Longitude coordinate
* 20 - sqft_living15: Living room area in 2015(implies-- some renovations)
* 21 - sqft_lot15: lotSize area in 2015(implies-- some renovations)

In [ ]:
df.columns

In [ ]:
df.describe().T

In [ ]:
df.describe(include='object').T

#### 2.1.5 Check for missing

In [ ]:
df.info()

#### 2.1.6 Check for duplicates

In [ ]:
df.duplicated().sum()

#### 2.1.7 Check for outliers

In [ ]:
feature_df = df.select_dtypes(include='number')

In [ ]:
rows, cols = 4, 5
num_columns = len(feature_df.columns)
num_plots = min(rows * cols, num_columns)

In [ ]:
fig, axes = plt.subplots(rows, cols, figsize=(12, 12))
axes = axes.flatten()

for i, column in enumerate(feature_df.columns[:rows * cols]):
    feature_df.boxplot(column=column, ax=axes[i])
    axes[i].set_title(f'Boxplot of {column}')

for j in range(num_plots, rows * cols):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

### 2.2 Exploratory data analysis

#### 2.2.1 Check distribution of numeric variables

In [ ]:
rows, cols = 4, 5
fig, axes = plt.subplots(rows, cols, figsize=(12, 12))
axes = axes.flatten()

# Select only numeric columns for histogram plotting
numeric_feature_df = feature_df.select_dtypes(include=np.number)
num_numeric_columns = len(numeric_feature_df.columns)
num_plots = min(rows * cols, num_numeric_columns)


for i, column in enumerate(numeric_feature_df.columns[:rows * cols]):
    numeric_feature_df.hist(column=column, ax=axes[i])
    axes[i].set_title(f'Histogram for {column}')

for j in range(num_plots, rows * cols):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

#### 2.2.2 Check skewness of numeric variables

In [ ]:
pd.set_option("display.max_rows", None)

In [ ]:
numeric_feature_df.skew().to_frame().rename(columns={0:"Feature Skewness"})

#### 2.2.3 Check correlation of numeric variables

##### 2.2.3.1 Correlation matrix

In [ ]:
fig =plt.figure(figsize=[12,12])
data_ploting = feature_df.corr(method= 'pearson')
sns.heatmap(data_ploting, cmap='Blues', linecolor='black', linewidths= 2, annot=True, fmt=".1f")
plt.show()

* 1 - sqft_above is correlated with sqft_living (0.9)
* 2 - bathrooms is correlated with sqft_living (0.8)
* 3 - sqft_living15 is correlated with sqft_living (0.8)
* 4 - grade is correlated with sqft_living (0.8)
* 5 - grade is correlated with sqft_above (0.8)

##### 2.2.3.2 Correlation analysis

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns

corr = df[num_cols].corrwith(df["price"]).sort_values(ascending=False)

sns.barplot(x=corr.values, y=corr.index)
plt.title("Correlation with Price")
plt.show()

#### 2.2.4 Check target distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 5))
axes = axes.flatten()

sns.histplot(df["price"], kde=True, ax=axes[0])

sns.boxplot(x=df["price"], ax=axes[1])

sns.ecdfplot(df["price"], ax=axes[2])

skewness = df["price"].skew()
kurtosis = df["price"].kurtosis()

axes[0].set_title("Price Distribution")
axes[1].set_title("Price Boxplot")
axes[2].set_title("Price ECDF")
axes[3].axis("off")  
axes[3].text(0.5, 0.6,f"Skewness: {skewness:.3f}",ha="center",fontsize=12,color="#00E5FF")
axes[3].text(0.5, 0.4,f"Kurtosis: {kurtosis:.3f}",ha="center",fontsize=12,color="#00E5FF")
axes[3].set_title("Summary Statistics")

plt.tight_layout()
plt.show()

#### 2.2.5 Bar chart of classification target

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(data=df, x="grade", palette="viridis")

plt.title("Distribution of House Grades in Dataset")
plt.xlabel("Grade")
plt.ylabel("Count")
plt.xticks(sorted(df["grade"].unique()))
plt.tight_layout()
plt.show()

#### 2.2.6 Inspect Observations with no bedrooms or bathrooms

In [ ]:
df[df["bedrooms"] <= 0].head(20)

In [ ]:
df[df["bathrooms"] <= 0].head(20)

## 3.0 Data preprocessing

### 3.1 Data cleansing

In [ ]:
df_clean = df.copy()
df_clean.drop_duplicates(inplace=True)

In [ ]:
df_clean.shape

#### 3.1.1 No missing values

#### 3.1.2 No duplicates

#### 3.1.3 Parse date

In [ ]:
df_clean["date"] = pd.to_datetime(df_clean["date"], errors="coerce")
df_clean["year_sold"] = df_clean["date"].dt.year
df_clean["month_sold"] = df_clean["date"].dt.month
df_clean.drop("date", axis=1, inplace=True)

In [ ]:
df_clean.shape

In [ ]:
df_final = df_clean.copy()
#df_final = df_clean.sample(frac=0.20, random_state=42)

In [ ]:
df_final.shape

#### 3.1.4 Row-wise feature engineering

In [ ]:
df_final["house_age"] = df_final["year_sold"] - df_final["yr_built"]
df_final["was_renovated"] = (df_final["yr_renovated"] > 0).astype(int)
df_final["has_basement"] = (df_final["sqft_basement"] > 0).astype(int)
df_final["total_rooms"] = (df_final["bedrooms"] + df_final["bathrooms"])
df_final["living_renovated"] = (df_final["sqft_living"] + df_final["sqft_living15"])
df_final["lot_renovated"] = (df_final["sqft_lot"] + df_final["sqft_lot15"])

In [ ]:
df_final.shape

In [ ]:
df_final.info()

### 3.2 First data partitioning (train, test, and validation split)

#### 3.2.1 Create log price target

In [ ]:
df_final["log_price"] = np.log1p(df_final["price"])

#### 3.2.2 Train test split

In [ ]:
X = df_final.drop(columns=["id", "price", "log_price"]) 
y = df_final["log_price"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42)

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train,test_size=0.1,random_state=42)

In [ ]:
print("Shape of x_train is:",X_train.shape)
print("Shape of x_valid is: ",X_valid.shape)
print("Shape of x_test is: ",X_test.shape)
print("Shape of y_train is:",y_train.shape)
print("Shape of y_valid is: ",y_valid.shape)
print("Shape of y_test is: ",y_test.shape)

In [ ]:
X_train.columns

### 3.3 Geographic feature engineering

#### 3.3.1 Geographic features

##### 3.3.1.1 Geographic centers

In [ ]:
# Data centroid
city_center_lat = X_train['lat'].mean()
city_center_lon = X_train['long'].mean()

# Seattle city center
seattle_center_lat = 47.6097
seattle_center_lon = -122.3331

In [ ]:
plt.figure(figsize=(10, 10))

sc = plt.scatter(
    df_final['long'],
    df_final['lat'],
    c=df_final['log_price'],
    cmap='turbo',
    s=10,
    alpha=0.6
)

cbar = plt.colorbar(sc)
cbar.set_label("Log-Price")

plt.scatter(
    city_center_lon,
    city_center_lat,
    color='red',
    marker='*',
    s=300,
    label='Data Centroid'
)

plt.scatter(
    seattle_center_lon,
    seattle_center_lat,
    color='blue',
    marker='*',
    s=300,
    label='Seattle Center'
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Seattle Housing Locations Colored by Log-Price")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))


In [ ]:
X_train['dist_centroid'] = haversine(
    X_train['lat'], X_train['long'],
    city_center_lat, city_center_lon
)

X_valid['dist_centroid'] = haversine(
    X_valid['lat'], X_valid['long'],
    city_center_lat, city_center_lon
)

X_test['dist_centroid'] = haversine(
    X_test['lat'], X_test['long'],
    city_center_lat, city_center_lon
)

X_train['dist_seattle_center'] = haversine(
    X_train['lat'], X_train['long'],
    seattle_center_lat, seattle_center_lon
)

X_valid['dist_seattle_center'] = haversine(
    X_valid['lat'], X_valid['long'],
    seattle_center_lat, seattle_center_lon
)

X_test['dist_seattle_center'] = haversine(
    X_test['lat'], X_test['long'],
    seattle_center_lat, seattle_center_lon
)


##### 3.3.1.2 Neighborhood clusters

In [ ]:
k = 12

In [ ]:
kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init='auto'
)

In [ ]:
kmeans.fit(X_train[['lat', 'long']])

In [ ]:
X_train['cluster'] = kmeans.predict(X_train[['lat', 'long']])
X_valid['cluster'] = kmeans.predict(X_valid[['lat', 'long']])
X_test['cluster'] = kmeans.predict(X_test[['lat', 'long']])

In [ ]:
plt.figure(figsize=(10, 10))

plt.scatter(
    df_final['long'],
    df_final['lat'],
    c=kmeans.predict(df_final[['lat', 'long']]),
    cmap='tab20',
    s=10,
    alpha=0.6
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("KMeans Neighborhood Clusters")
plt.grid(True)
plt.show()


##### 3.3.1.3 Distance to water

In [ ]:
def point_to_box_distance(lat, lon, lat_min, lat_max, lon_min, lon_max):
    clamped_lat = np.clip(lat, lat_min, lat_max)
    clamped_lon = np.clip(lon, lon_min, lon_max)
    return haversine(lat, lon, clamped_lat, clamped_lon)

In [ ]:
# Distance to Lake Washington (approximate bounding box)
X_train['dist_lake_washington'] = point_to_box_distance(
    X_train['lat'], X_train['long'],
    47.50, 47.75, -122.30, -122.20
)

X_valid['dist_lake_washington'] = point_to_box_distance(
    X_valid['lat'], X_valid['long'],
    47.50, 47.75, -122.30, -122.20
)

X_test['dist_lake_washington'] = point_to_box_distance(
    X_test['lat'], X_test['long'],
    47.50, 47.75, -122.30, -122.20
)

# Distance to Puget Sound (approximate bounding box)
X_train['dist_puget_sound'] = point_to_box_distance(
    X_train['lat'], X_train['long'],
    47.45, 47.80, -122.55, -122.35
)

X_valid['dist_puget_sound'] = point_to_box_distance(
    X_valid['lat'], X_valid['long'],
    47.45, 47.80, -122.55, -122.35
)

X_test['dist_puget_sound'] = point_to_box_distance(
    X_test['lat'], X_test['long'],
    47.45, 47.80, -122.55, -122.35
)

# Minimum distance to any water body
X_train['dist_water'] = np.minimum(
    X_train['dist_lake_washington'],
    X_train['dist_puget_sound']
)

X_valid['dist_water'] = np.minimum(
    X_valid['dist_lake_washington'],
    X_valid['dist_puget_sound']
)

X_test['dist_water'] = np.minimum(
    X_test['dist_lake_washington'],
    X_test['dist_puget_sound']
)


In [ ]:
plt.figure(figsize=(10, 10))

plt.scatter(
    X_train['long'],
    X_train['lat'],
    c=X_train['dist_water'],
    cmap='turbo',
    s=10,
    alpha=0.6
)

cbar = plt.colorbar()
cbar.set_label("Distance to Water (km)")

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Seattle Housing Locations Colored by Distance to Water")
plt.grid(True)
plt.show()


### 3.4 Second data partitioning (regression and classification split)

#### 3.4.1 Regression, classification split

In [ ]:
y_class = df_final["grade"]

In [ ]:
y_class_train = y_class.loc[X_train.index]
y_class_valid = y_class.loc[X_valid.index]
y_class_test  = y_class.loc[X_test.index]

In [ ]:
X_reg_train = X_train.copy()
X_reg_valid = X_valid.copy()
X_reg_test = X_test.copy()

In [ ]:
X_class_train = X_train.drop(columns=["grade"])
X_class_valid = X_valid.drop(columns=["grade"])
X_class_test = X_test.drop(columns=["grade"])

In [ ]:
y_reg_train = y_train
y_reg_valid = y_valid
y_reg_test  = y_test

### 3.5 Interaction feature engineering

#### 3.5.1 Regression interaction feature engineering

In [ ]:
# sqft_living × grade
X_reg_train['int_sqft_grade'] = X_reg_train['sqft_living'] * X_reg_train['grade']
X_reg_valid['int_sqft_grade'] = X_reg_valid['sqft_living'] * X_reg_valid['grade']
X_reg_test['int_sqft_grade'] = X_reg_test['sqft_living'] * X_reg_test['grade']

# condition × age
X_reg_train['int_condition_age'] = X_reg_train['condition'] * X_reg_train['house_age']
X_reg_valid['int_condition_age'] = X_reg_valid['condition'] * X_reg_valid['house_age']
X_reg_test['int_condition_age'] = X_reg_test['condition'] * X_reg_test['house_age']

# bedrooms × bathrooms
X_reg_train['int_bed_bath'] = X_reg_train['bedrooms'] * X_reg_train['bathrooms']
X_reg_valid['int_bed_bath'] = X_reg_valid['bedrooms'] * X_reg_valid['bathrooms']
X_reg_test['int_bed_bath'] = X_reg_test['bedrooms'] * X_reg_test['bathrooms']

# latitude × distance_to_center
X_reg_train['int_lat_dist_center'] = X_reg_train['lat'] * X_reg_train['dist_seattle_center']
X_reg_valid['int_lat_dist_center'] = X_reg_valid['lat'] * X_reg_valid['dist_seattle_center']
X_reg_test['int_lat_dist_center'] = X_reg_test['lat'] * X_reg_test['dist_seattle_center']

# longitude × distance_to_centroid
X_reg_train['int_long_dist_centroid'] = X_reg_train['long'] * X_reg_train['dist_centroid']
X_reg_valid['int_long_dist_centroid'] = X_reg_valid['long'] * X_reg_valid['dist_centroid']
X_reg_test['int_long_dist_centroid'] = X_reg_test['long'] * X_reg_test['dist_centroid']

# view × distance_to_water
X_reg_train['int_view_dist_water'] = X_reg_train['view'] * X_reg_train['dist_water']
X_reg_valid['int_view_dist_water'] = X_reg_valid['view'] * X_reg_valid['dist_water']
X_reg_test['int_view_dist_water'] = X_reg_test['view'] * X_reg_test['dist_water']

#### 3.5.2 Classification interaction features

In [ ]:
# condition × age
X_class_train['int_condition_age'] = X_class_train['condition'] * X_class_train['house_age']
X_class_valid['int_condition_age'] = X_class_valid['condition'] * X_class_valid['house_age']
X_class_test['int_condition_age'] = X_class_test['condition'] * X_class_test['house_age']

# bedrooms × bathrooms
X_class_train['int_bed_bath'] = X_class_train['bedrooms'] * X_class_train['bathrooms']
X_class_valid['int_bed_bath'] = X_class_valid['bedrooms'] * X_class_valid['bathrooms']
X_class_test['int_bed_bath'] = X_class_test['bedrooms'] * X_class_test['bathrooms']

# latitude × distance_to_center
X_class_train['int_lat_dist_center'] = X_class_train['lat'] * X_class_train['dist_seattle_center']
X_class_valid['int_lat_dist_center'] = X_class_valid['lat'] * X_class_valid['dist_seattle_center']
X_class_test['int_lat_dist_center'] = X_class_test['lat'] * X_class_test['dist_seattle_center']

# longitude × distance_to_centroid
X_class_train['int_long_dist_centroid'] = X_class_train['long'] * X_class_train['dist_centroid']
X_class_valid['int_long_dist_centroid'] = X_class_valid['long'] * X_class_valid['dist_centroid']
X_class_test['int_long_dist_centroid'] = X_class_test['long'] * X_class_test['dist_centroid']

# view × distance_to_water
X_class_train['int_view_dist_water'] = X_class_train['view'] * X_class_train['dist_water']
X_class_valid['int_view_dist_water'] = X_class_valid['view'] * X_class_valid['dist_water']
X_class_test['int_view_dist_water'] = X_class_test['view'] * X_class_test['dist_water']

### 3.6 Encoding, normalization, and scaling of linear and logistic regression data

#### 3.6.1 One hot encode nominal categorical features for linear and logistic regression

In [ ]:
categorical_nominal = ['cluster', 'zipcode']

In [ ]:
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(X_class_train[categorical_nominal])

In [ ]:
X_class_train_encoded = encoder.transform(X_class_train[categorical_nominal])
X_class_valid_encoded = encoder.transform(X_class_valid[categorical_nominal])
X_class_test_encoded  = encoder.transform(X_class_test[categorical_nominal])

In [ ]:
encoded_cols = list(encoder.get_feature_names_out())

#### 3.6.2 Remove multicollinear numeric features for linear and logistic regression

In [ ]:
remove_numeric = ["sqft_above", "sqft_living15", "bathrooms"]

In [ ]:
remove_class_only = ["grade", "int_sqft_grade"]

In [ ]:
geo_numeric = ["lat", "long"]

In [ ]:
numeric_reg = X_reg_train.drop(columns=categorical_nominal).columns
numeric_class = X_class_train.drop(columns=categorical_nominal).columns

In [ ]:
numeric_reg = [col for col in numeric_reg 
               if col not in remove_numeric + categorical_nominal + geo_numeric]

numeric_class = [col for col in numeric_class 
                 if col not in remove_numeric + remove_class_only + geo_numeric]


In [ ]:
print(numeric_reg)

#### 3.6.3 Normalize numeric features for linear and logistic regression

In [ ]:
normalizer = PowerTransformer(method="yeo-johnson")
normalizer.fit(X_reg_train[numeric_reg])

In [ ]:
X_reg_train_norm = X_reg_train.copy()
X_reg_valid_norm = X_reg_valid.copy()
X_reg_test_norm  = X_reg_test.copy()

X_reg_train_norm[numeric_reg] = normalizer.transform(X_reg_train[numeric_reg])
X_reg_valid_norm[numeric_reg] = normalizer.transform(X_reg_valid[numeric_reg])
X_reg_test_norm[numeric_reg]  = normalizer.transform(X_reg_test[numeric_reg])


In [ ]:
normalizer.fit(X_class_train[numeric_class])

In [ ]:
X_class_train_norm = X_class_train.copy()
X_class_valid_norm = X_class_valid.copy()
X_class_test_norm  = X_class_test.copy()

X_class_train_norm[numeric_class] = normalizer.transform(X_class_train[numeric_class])
X_class_valid_norm[numeric_class] = normalizer.transform(X_class_valid[numeric_class])
X_class_test_norm[numeric_class]  = normalizer.transform(X_class_test[numeric_class])

In [ ]:
X_reg_train_norm[geo_numeric] = X_reg_train[geo_numeric]
X_reg_valid_norm[geo_numeric] = X_reg_valid[geo_numeric]
X_reg_test_norm[geo_numeric]  = X_reg_test[geo_numeric]

X_class_train_norm[geo_numeric] = X_class_train[geo_numeric]
X_class_valid_norm[geo_numeric] = X_class_valid[geo_numeric]
X_class_test_norm[geo_numeric]  = X_class_test[geo_numeric]

#### 3.6.4 Scale numeric features for linear and logistic regression

In [ ]:
scaler_reg = StandardScaler()
scaler_reg.fit(X_reg_train_norm[numeric_reg])

scaler_class = StandardScaler()
scaler_class.fit(X_class_train_norm[numeric_class])

In [ ]:
X_reg_train_scaled_num = scaler_reg.transform(X_reg_train_norm[numeric_reg])
X_reg_valid_scaled_num = scaler_reg.transform(X_reg_valid_norm[numeric_reg])
X_reg_test_scaled_num  = scaler_reg.transform(X_reg_test_norm[numeric_reg])

In [ ]:
X_class_train_scaled_num = scaler_class.transform(X_class_train_norm[numeric_class])
X_class_valid_scaled_num = scaler_class.transform(X_class_valid_norm[numeric_class])
X_class_test_scaled_num  = scaler_class.transform(X_class_test_norm[numeric_class])

#### 3.6.5 Merge scaled and encoded linear and logistic regression features

In [ ]:
X_reg_train_lr = X_reg_train_scaled_num.copy()
X_reg_valid_lr = X_reg_valid_scaled_num.copy()
X_reg_test_lr  = X_reg_test_scaled_num.copy()

In [ ]:
X_class_train_lr = np.hstack([X_class_train_scaled_num, X_class_train_encoded])
X_class_valid_lr = np.hstack([X_class_valid_scaled_num, X_class_valid_encoded])
X_class_test_lr  = np.hstack([X_class_test_scaled_num,  X_class_test_encoded])

In [ ]:
print("X_reg_train_lr shape:", X_reg_train_lr.shape)
print("X_class_train_lr shape:", X_class_train_lr.shape)

### 3.7 Feature sets

#### 3.7.1 Base model features

In [ ]:
baseline_features = [
    'bedrooms', 
    'bathrooms', 
    'sqft_living',
    'sqft_lot', 
    'floors', 
    'waterfront', 
    'view', 
    'condition', 
    'grade',
    'sqft_above', 
    'sqft_basement', 
    'yr_built', 
    'yr_renovated', 
    'zipcode',
    'lat', 
    'long', 
    'sqft_living15', 
    'sqft_lot15',
    'year_sold', 
    'month_sold' 
]

#### 3.7.2 Engineered hedonic features

In [ ]:
hedonic_features = baseline_features + [
    'house_age', 
    'was_renovated', 
    'has_basement', 
    'total_rooms', 
    'living_renovated',
    'lot_renovated',
]

#### 3.7.3 Engineered geographic features

In [ ]:
geographic_features = hedonic_features + [
    'cluster',
    'dist_lake_washington',
    'dist_puget_sound',
    'dist_water',
    'dist_centroid',
    'dist_seattle_center',
]

#### 3.7.4 Engineered interaction features

In [ ]:
interaction_features = geographic_features + [
    'int_sqft_grade',
    'int_condition_age',
    'int_bed_bath',
    'int_lat_dist_center',
    'int_long_dist_centroid',
    'int_view_dist_water',
]

#### 3.7.5 Feature sets

In [ ]:
feature_sets = {
    "baseline": baseline_features,
    "hedonic": hedonic_features,
    "geographic": geographic_features,
    "interaction": interaction_features,
}

### 3.8 Convert linear and logistic regression matrices to data frames

In [ ]:
reg_lr_columns = list(numeric_reg) + encoded_cols
class_lr_columns = list(numeric_class) + encoded_cols

In [ ]:
def expand_feature_set(feature_names, numeric_cols, encoded_cols):
    expanded = []
    for f in feature_names:
        if f in numeric_cols:
            expanded.append(f)
        elif f == "zipcode":
            expanded.extend([c for c in encoded_cols if c.startswith("zipcode_")])
        elif f == "cluster":
            expanded.extend([c for c in encoded_cols if c.startswith("cluster_")])
    return expanded

In [ ]:
feature_sets_reg = feature_sets.copy()

In [ ]:
feature_sets_class = {
    name: [f for f in features if f != "grade"]
    for name, features in feature_sets.items()
}

In [ ]:
feature_sets_reg_lr = {
    "baseline": numeric_reg,
    "hedonic": numeric_reg,
    "geographic": numeric_reg,
    "interaction": numeric_reg
}

In [ ]:
feature_sets_class_lr = {
    name: expand_feature_set(features, numeric_class, encoded_cols)
    for name, features in feature_sets_class.items()
}

In [ ]:
X_reg_train_lr_dict = {}
X_reg_valid_lr_dict = {}
X_reg_test_lr_dict = {}

for iteration_name, feature_names in feature_sets_reg_lr.items():
    idx = [reg_lr_columns.index(f) for f in feature_names]

    X_reg_train_lr_dict[iteration_name] = pd.DataFrame(
        X_reg_train_lr[:, idx],
        columns=feature_names
    )

    X_reg_valid_lr_dict[iteration_name] = pd.DataFrame(
        X_reg_valid_lr[:, idx],
        columns=feature_names
    )

    X_reg_test_lr_dict[iteration_name] = pd.DataFrame(
        X_reg_test_lr[:, idx],
        columns=feature_names
    )

In [ ]:
X_class_train_lr_dict = {}
X_class_valid_lr_dict = {}
X_class_test_lr_dict = {}

for iteration_name, feature_names in feature_sets_class_lr.items():
    idx = [class_lr_columns.index(f) for f in feature_names]

    X_class_train_lr_dict[iteration_name] = pd.DataFrame(
        X_class_train_lr[:, idx],
        columns=feature_names
    )

    X_class_valid_lr_dict[iteration_name] = pd.DataFrame(
        X_class_valid_lr[:, idx],
        columns=feature_names
    )

    X_class_test_lr_dict[iteration_name] = pd.DataFrame(
        X_class_test_lr[:, idx],
        columns=feature_names
    )

### 3.9 Feature set registry

#### 3.9.1 Regression tree feature set

In [ ]:
feature_sets_reg = {
    name: features.copy()
    for name, features in feature_sets.items()
}

#### 3.9.2 Classification tree feature sets

In [ ]:
feature_sets_class = {
    name: [
        f for f in features
        if f != "grade"
        and not f.startswith("grade_")
        and f != "int_sqft_grade"
    ]
    for name, features in feature_sets.items()
}

#### 3.9.3 Linear regression feature sets

In [ ]:
feature_sets_reg_lr = {
    name: expand_feature_set(
        [f for f in features if f not in remove_numeric],
        numeric_reg,
        encoded_cols
    )
    for name, features in feature_sets_reg.items()
}

#### 3.9.4 Logistic regression feature sets

In [ ]:
feature_sets_class_lr = {
    name: expand_feature_set(
        [
            f for f in features
            if f not in remove_numeric
            and f not in ["grade", "int_sqft_grade"]
        ],
        numeric_class,
        encoded_cols
    )
    for name, features in feature_sets_class.items()
}

#### 3.9.5 Feature registry

In [ ]:
FEATURE_REGISTRY = {
    "reg_tree": feature_sets_reg,
    "class_tree": feature_sets_class,
    "reg_lr": feature_sets_reg_lr,
    "class_lr": feature_sets_class_lr
}

## 4.0 Modeling

In [ ]:
MODEL_FAMILY = {
    "LinearRegression": "reg_lr",
    "CatBoostRegressor": "reg_tree",
    "RandomForestRegressor": "reg_tree",

    "LogisticRegression": "class_lr",
    "CatBoostClassifier": "class_tree",
    "RandomForestClassifier": "class_tree"
}

### 4.1 Regression models

In [ ]:
def regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    
    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    }

In [ ]:
def evaluate_regression(model, X_train, y_train, X_valid, y_valid):
    y_pred_train = model.predict(X_train)
    y_pred_valid = model.predict(X_valid)

    metrics_train = regression_metrics(y_train, y_pred_train)
    metrics_valid = regression_metrics(y_valid, y_pred_valid)

    metrics = {f"{k}_train": v for k, v in metrics_train.items()}
    metrics.update({f"{k}_valid": v for k, v in metrics_valid.items()})

    return metrics

In [ ]:
regression_models = {
    "catboost": CatBoostRegressor,
    "random_forest": RandomForestRegressor,
    "linear_regression": LinearRegression
}

### 4.2 Classification models

In [ ]:
def classification_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='macro')
    recall = recall_score(y_true, y_pred, average='macro')
    f1 = f1_score(y_true, y_pred, average='macro')
    
    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    }

In [ ]:
def evaluate_classification(model, X_train, y_train, X_valid, y_valid):
    y_pred_train = model.predict(X_train)
    y_pred_valid = model.predict(X_valid)

    metrics_train = classification_metrics(y_train, y_pred_train)
    metrics_valid = classification_metrics(y_valid, y_pred_valid)

    metrics = {f"{k}_train": v for k, v in metrics_train.items()}
    metrics.update({f"{k}_valid": v for k, v in metrics_valid.items()})

    return metrics

In [ ]:
classes = np.unique(y_class_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_class_train
)

class_weight_dict = {cls: w for cls, w in zip(classes, weights)}

catboost_class_weights = [class_weight_dict[c] for c in sorted(class_weight_dict.keys())]

In [ ]:
classification_models = {
    "catboost": CatBoostClassifier,
    "random_forest": RandomForestClassifier,
    "logistic_regression": LogisticRegression
}

### 4.3 Model iterations

#### 4.3.1 Regression model iterations

In [ ]:
results_regression = []

In [ ]:
for model_name, ModelClass in regression_models.items():

    family = MODEL_FAMILY[ModelClass.__name__]
    feature_sets = FEATURE_REGISTRY[family]

    for iteration_name, feature_names in feature_sets.items():
    
        if model_name == "catboost":
            model = ModelClass(
                iterations=1000,
                learning_rate=0.03,
                depth=6,
                loss_function="RMSE",
                verbose=0,
                od_type="Iter",
                od_wait=20
            )

        elif model_name == "random_forest":
            model = ModelClass(
                n_estimators=300,
                criterion="squared_error",
                max_depth=None,
                min_samples_split=2,
                min_samples_leaf=1,
                max_features="sqrt",
                bootstrap=True,
                random_state=42
            )

        else:  # linear regression
            model = ModelClass(
                fit_intercept=True,
                copy_X=True,
                n_jobs=-1
            )

        # Feature routing
        if model_name == "linear_regression":
            Xtr = X_reg_train_lr_dict[iteration_name]
            Xva = X_reg_valid_lr_dict[iteration_name]
        else:
            Xtr = X_reg_train[feature_names]
            Xva = X_reg_valid[feature_names]

        # CatBoost categorical handling
        if model_name == "catboost":
            cat_idx = [
                Xtr.columns.get_loc(c)
                for c in categorical_nominal
                if c in Xtr.columns
            ]

            model.fit(
                Xtr, y_reg_train,
                eval_set=(Xva, y_reg_valid),
                cat_features=cat_idx
            )

        else:
            model.fit(Xtr, y_reg_train)

        # Evaluate
        metrics = evaluate_regression(
            model=model,
            X_train=Xtr,
            y_train=y_reg_train,
            X_valid=Xva,
            y_valid=y_reg_valid
        )

        results_regression.append({
            "iteration": iteration_name,
            "model": model_name,
            "model_object": model,
            "features": feature_names,
            **metrics
        })

        print(f"[Regression] Iteration: {iteration_name} | Model: {model_name} → completed")


In [ ]:
type(X_reg_train_lr_dict['baseline'])

#### 4.3.2 Regression model results

In [ ]:
df_results_regression = pd.DataFrame(results_regression)

df_results_regression = df_results_regression.sort_values(
    by=["iteration", "RMSE_valid"], ascending=[True, True]
).reset_index(drop=True)

df_results_regression

#### 4.3.3 Best regression model

In [ ]:
df_results_regression = pd.DataFrame(results_regression)

df_results_regression = df_results_regression.sort_values(
    by=["RMSE_valid"], ascending=True
).reset_index(drop=True)

best_reg_row = df_results_regression.iloc[0]

best_reg_model = best_reg_row["model_object"]      
best_reg_features = best_reg_row["features"]       
best_reg_iteration = best_reg_row["iteration"]     
best_reg_model_name = best_reg_row["model"]  

print("Best Regression Model:")
print(f"Iteration: {best_reg_iteration}")
print(f"Model: {best_reg_model_name}")
print(best_reg_row)

In [ ]:
print(f"Best regression features: {best_reg_features}")

In [ ]:
Xtr_reg = X_reg_train[best_reg_features]
Xva_reg = X_reg_valid[best_reg_features]

In [ ]:
best_reg_cat_idx = [
    i for i, col in enumerate(best_reg_features)
    if col in categorical_nominal
]

In [ ]:
def fit_best_regression_model(model, Xtr, ytr, Xva=None, yva=None, cat_idx=None):

    if isinstance(model, CatBoostRegressor):
        return model.fit(
            Xtr, ytr,
            eval_set=(Xva, yva) if Xva is not None else None,
            cat_features=cat_idx,
            verbose=False
        )

    # All other models
    return model.fit(Xtr, ytr)


In [ ]:
fit_best_regression_model(best_reg_model, Xtr_reg, y_reg_train, Xva_reg, y_reg_valid, cat_idx=best_reg_cat_idx)

In [ ]:
if "catboost" in best_reg_model.__class__.__name__.lower():
    best_reg_model.save_model("catboost_regression_best.cbm")
else:
    joblib.dump(best_reg_model, "best_regression_model.pkl")

##### 4.3.3.1 Best regression model diagnostics

In [ ]:
y_pred_log = best_reg_model.predict(Xva_reg)
y_true_log = y_reg_valid
residuals = y_true_log - y_pred_log

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Residual Distribution
axes[0, 0].hist(residuals, bins=50, color='steelblue', edgecolor='black')
axes[0, 0].set_title("Residual Distribution (Validation)")
axes[0, 0].set_xlabel("Residual")
axes[0, 0].set_ylabel("Frequency")

# 2. Residuals vs Predicted
axes[0, 1].scatter(y_pred_log, residuals, alpha=0.5)
axes[0, 1].axhline(0, color='red', linestyle='--')
axes[0, 1].set_title("Residuals vs Predicted (Validation)")
axes[0, 1].set_xlabel("Predicted (log)")
axes[0, 1].set_ylabel("Residuals")

# 3. Q-Q Plot
stats.probplot(residuals, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title("Q-Q Plot (Validation)")

# 4. Actual vs Predicted
axes[1, 1].scatter(y_true_log, y_pred_log, alpha=0.5)

min_val = min(y_true_log.min(), y_pred_log.min())
max_val = max(y_true_log.max(), y_pred_log.max())
axes[1, 1].plot([min_val, max_val], [min_val, max_val], color='red')

axes[1, 1].set_xlabel("Actual (log)")
axes[1, 1].set_ylabel("Predicted (log)")
axes[1, 1].set_title("Actual vs Predicted (Validation)")

plt.tight_layout()
plt.show()


#### 4.3.4 Classification model iterations

In [ ]:
results_classification = []

In [ ]:
feature_sets_classification = {
    name: [f for f in feats if f != "grade"]
    for name, feats in feature_sets.items()
}

In [ ]:
print("Unique classes in y_class_train:", np.unique(y_class_train))

In [ ]:
print("class_tree features:", FEATURE_REGISTRY["class_tree"])


In [ ]:
for model_name, ModelClass in classification_models.items():

    family = MODEL_FAMILY[ModelClass.__name__]
    feature_sets = FEATURE_REGISTRY[family]

    for iteration_name, feature_names in feature_sets.items():

        if model_name == "catboost":
            model = ModelClass(
                iterations=1000,
                learning_rate=0.03,
                depth=6,
                loss_function="MultiClass",
                verbose=0,
                od_type="Iter",
                od_wait=20
            )

        elif model_name == "random_forest":
            model = ModelClass(
                n_estimators=100,
                criterion="gini",
                max_depth=None,
                min_samples_split=2,
                min_samples_leaf=1,
                max_features="sqrt",
                bootstrap=True,
                random_state=42
            )

        else:  
            model = ModelClass(
                penalty="l2",
                C=1.0,
                solver="lbfgs",
                max_iter=1000,
                fit_intercept=True
            )

        if model_name == "catboost":
            model.set_params(class_weights=catboost_class_weights)

        else:
            model.set_params(class_weight=class_weight_dict)

        if model_name == "logistic_regression":
            Xtr = X_class_train_lr_dict[iteration_name]
            Xva = X_class_valid_lr_dict[iteration_name]
        else:
            Xtr = X_class_train[feature_names]
            Xva = X_class_valid[feature_names]

        if model_name == "catboost":
            cat_idx = [
                Xtr.columns.get_loc(c)
                for c in categorical_nominal
                if c in Xtr.columns
            ]

            model.fit(
                Xtr, y_class_train,
                eval_set=(Xva, y_class_valid),
                cat_features=cat_idx,
                verbose=False
            )

        else:
            model.fit(Xtr, y_class_train)

        metrics = evaluate_classification(
            model=model,
            X_train=Xtr,
            y_train=y_class_train,
            X_valid=Xva,
            y_valid=y_class_valid
        )

        results_classification.append({
            "iteration": iteration_name,
            "model_name": model_name,
            "model_object": model,
            "features": feature_names,   
            **metrics
        })

        print(f"[Classification] Iteration: {iteration_name} | Model: {model_name} → completed")


#### 4.3.5 Classification model results

In [ ]:
df_results_classification = pd.DataFrame(results_classification)

df_results_classification = df_results_classification.sort_values(
    by=["iteration", "Accuracy_valid"], ascending=[True, False]
).reset_index(drop=True)

df_results_classification

#### 4.3.6 Best classification model

In [ ]:
df_results_classification = pd.DataFrame(results_classification)

df_results_classification = df_results_classification.sort_values(
    by=["Accuracy_valid"], ascending=False
).reset_index(drop=True)

best_class_row = df_results_classification.iloc[0]

best_class_model      = best_class_row["model_object"]  
best_class_features   = best_class_row["features"]     
best_class_iteration  = best_class_row["iteration"]
best_class_model_name = best_class_row["model_name"]

print("Best Classification Model:")
print(f"Iteration: {best_class_iteration}")
print(f"Model: {best_class_model_name}")
print(best_class_row)

In [ ]:
print(f"Best class features: {best_class_features}")

In [ ]:
Xtr_class = X_class_train[best_class_features]
Xva_class = X_class_valid[best_class_features]

In [ ]:
best_class_cat_idx = [
    i for i, col in enumerate(best_class_features)
    if col in categorical_nominal
]


In [ ]:
def fit_best_model(model, Xtr, ytr, Xva=None, yva=None, cat_idx=None):

    model_name = model.__class__.__name__.lower()

    if "catboost" in model_name:
        return model.fit(
            Xtr, ytr,
            eval_set=(Xva, yva) if Xva is not None else None,
            cat_features=cat_idx,
            verbose=False
        )

    return model.fit(Xtr, ytr)

In [ ]:
fit_best_model(best_class_model, Xtr_class, y_class_train, Xva_class, y_class_valid, cat_idx=best_class_cat_idx)

In [ ]:
if "catboost" in best_class_model.__class__.__name__.lower():
    best_class_model.save_model("catboost_classification_best.cbm")
else:
    joblib.dump(best_class_model, "best_classification_model.pkl")


##### 4.3.6.1 Best classification model diagnostics

In [ ]:
# --- Predict on validation set ---
preds_valid = best_class_model.predict(Xva_class[best_class_features])
probs_valid = best_class_model.predict_proba(Xva_class[best_class_features])

y_true = y_class_valid
classes = sorted(y_true.unique())

# Binarize for ROC curves
y_valid_bin = label_binarize(y_true, classes=classes)

# Confusion matrix
cm = confusion_matrix(y_true, preds_valid)

# Features used
features_used = best_class_features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
# ---------------------------------------------------------
# 1. Confusion Matrix
# ---------------------------------------------------------
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=classes,
    yticklabels=classes,
    ax=axes[0, 0]
)
axes[0, 0].set_title("Confusion Matrix (Validation)")
axes[0, 0].set_xlabel("Predicted")
axes[0, 0].set_ylabel("Actual")

# ---------------------------------------------------------
# 2. ROC Curves (One-vs-Rest)
# ---------------------------------------------------------
for i, cls in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_valid_bin[:, i], probs_valid[:, i])
    roc_auc_i = auc(fpr, tpr)
    axes[0, 1].plot(fpr, tpr, label=f"Class {cls} (AUC={roc_auc_i:.3f})")

axes[0, 1].plot([0, 1], [0, 1], "k--")
axes[0, 1].set_title("ROC Curves (Validation)")
axes[0, 1].set_xlabel("False Positive Rate")
axes[0, 1].set_ylabel("True Positive Rate")
axes[0, 1].legend()

# ---------------------------------------------------------
# 3. Classification Report
# ---------------------------------------------------------
report = classification_report(y_true, preds_valid)
axes[1, 0].axis("off")
axes[1, 0].text(
    0.01, 0.99,
    report,
    fontsize=12,
    family="monospace",
    va="top"
)
axes[1, 0].set_title("Classification Report (Validation)")

# ---------------------------------------------------------
# 4. Features Used
# ---------------------------------------------------------
axes[1, 1].axis("off")
axes[1, 1].text(
    0.01, 0.99,
    "Features Used:\n" + "\n".join(features_used),
    fontsize=12,
    family="monospace",
    va="top"
)
axes[1, 1].set_title("Feature Set Used")

plt.tight_layout()
plt.show()


### 4.4 Best model feature importance

#### 4.4.1 Regression feature importance

In [ ]:
def get_regression_importance(model, feature_names):
    name = model.__class__.__name__.lower()

    # --- CatBoostRegressor ---
    if "catboost" in name:
        importances = model.get_feature_importance(type="PredictionValuesChange")
        return list(zip(feature_names, importances))

    # --- RandomForestRegressor ---
    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
        return list(zip(feature_names, importances))

    # --- LinearRegression ---
    if hasattr(model, "coef_"):
        # coef_ shape: (n_features,) for regression
        importances = np.abs(model.coef_)
        return list(zip(feature_names, importances))

    raise ValueError(f"Model type {model.__class__.__name__} not supported.")


In [ ]:
importances_reg = get_regression_importance(best_reg_model, best_reg_features)

In [ ]:
print(f"Regression feature importances: {importances_reg}")

In [ ]:
importances_reg_sorted = sorted(importances_reg, key=lambda x: x[1], reverse=True)

In [ ]:
print(f"Sorted regression feature importances: {importances_reg_sorted}")

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh([f[0] for f in importances_reg_sorted], [f[1] for f in importances_reg_sorted])
plt.gca().invert_yaxis()
plt.title("CatBoost Regression Feature Importance")
plt.xlabel("Importance")
plt.show()

#### 4.4.2 Classification feature importance

In [ ]:
def get_model_importance(model, feature_names):
    name = model.__class__.__name__.lower()

    # --- CatBoost ---
    if "catboost" in name:
        importances = model.get_feature_importance(type="PredictionValuesChange")
        return list(zip(feature_names, importances))

    # --- RandomForest ---
    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
        return list(zip(feature_names, importances))

    # --- Logistic Regression ---
    if hasattr(model, "coef_"):
        # For binary classification, coef_ is shape (1, n_features)
        importances = np.abs(model.coef_[0])
        return list(zip(feature_names, importances))

    raise ValueError(f"Model type {model.__class__.__name__} not supported.")


In [ ]:
importances_class = get_model_importance(best_class_model, best_class_features)

In [ ]:
print(f"Classification feature importances: {importances_class}")

In [ ]:
importances_class_sorted = sorted(importances_class, key=lambda x: x[1], reverse=True)

In [ ]:
print(f"Sorted classification feature importances: {importances_class_sorted}")

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh([f[0] for f in importances_class_sorted], [f[1] for f in importances_class_sorted])
plt.gca().invert_yaxis()
plt.title("CatBoost Classification Feature Importance")
plt.xlabel("Importance")
plt.show()

### 4.5 Hyperparameter optimization

#### 4.5.1 Regression hyperparameter optimization

##### 4.5.1.1 Regression hyperparameter distribution

In [ ]:
reg_param_distributions = {
    "CatBoostRegressor": {
        "depth": [4, 6, 8, 10],
        "learning_rate": np.linspace(0.01, 0.2, 10),
        "iterations": [300, 500, 800, 1200]
    },

    "RandomForestRegressor": {
        "n_estimators": [200, 400, 800, 1200],
        "max_depth": [None, 5, 10, 20],
        "max_features": ["sqrt", "log2", 0.3, 0.5]
    },

    "LinearRegression": {
        "fit_intercept": [True, False],
        "copy_X": [True, False],
        "positive": [True, False]
    }
}


In [ ]:
best_reg_model_name = best_reg_model.__class__.__name__
reg_search_space = reg_param_distributions[best_reg_model_name]

print(f"Regression hyperparameter search space for {best_reg_model_name}:")
print(reg_search_space)

##### 4.5.1.2 Regression hyperparameter tuning

In [ ]:
best_reg_row = df_results_regression.loc[
    df_results_regression['RMSE_valid'].idxmin()
]

In [ ]:
best_reg_features = best_reg_row["features"]
best_reg_iteration = best_reg_row["iteration"]

In [ ]:
Xtr_reg_best = X_reg_train[best_reg_features]
Xva_reg_best = X_reg_valid[best_reg_features]

In [ ]:
reg_estimators_to_tune = {
    "CatBoostRegressor": CatBoostRegressor(
        verbose=0,
        random_state=42,
        od_type="Iter",
        od_wait=20
    ),
    "RandomForestRegressor": RandomForestRegressor(
        random_state=42
    )
}

In [ ]:
reg_hpo_results = {}

In [ ]:
for model_name, base_estimator in reg_estimators_to_tune.items():
    print(f"\nRunning HPO for {model_name}...")

    search_space = reg_param_distributions[model_name]

    reg_search = RandomizedSearchCV(
        estimator=base_estimator,
        param_distributions=search_space,
        n_iter=10,   # increase later
        scoring="neg_root_mean_squared_error",
        cv=5,       # increase later
        n_jobs=-1,
        random_state=42,
        refit=True
    )

    reg_search.fit(Xtr_reg_best, y_reg_train)

    best_model_hpo = reg_search.best_estimator_
    best_params = reg_search.best_params_

    print(f"Best hyperparameters for {model_name}:")
    print(best_params)

    reg_hpo_results[model_name] = {
        "estimator": best_model_hpo,
        "params": best_params,
        "search_obj": reg_search
    }


In [ ]:
print("Features used during HPO:")
for f in best_reg_features:
    print("-", f)

##### 4.5.1.3 Regression hyperparameter tuning diagnostics

In [ ]:
for model_name, result in reg_hpo_results.items():
    print(f"\n=== Diagnostics for {model_name} ===")

    model = result["estimator"]

    y_pred = model.predict(Xva_reg_best)
    y_true = y_reg_valid
    residuals = y_true - y_pred

    fig, axes = plt.subplots(2, 2, figsize=(14, 12))

    # 1. Residual Distribution
    axes[0, 0].hist(residuals, bins=50, color='steelblue', edgecolor='black')
    axes[0, 0].set_title(f"Residual Distribution (Validation) - {model_name}")
    axes[0, 0].set_xlabel("Residual")
    axes[0, 0].set_ylabel("Frequency")

    # 2. Residuals vs Predicted
    axes[0, 1].scatter(y_pred, residuals, alpha=0.5)
    axes[0, 1].axhline(0, color='red', linestyle='--')
    axes[0, 1].set_title(f"Residuals vs Predicted (Validation) - {model_name}")
    axes[0, 1].set_xlabel("Predicted")
    axes[0, 1].set_ylabel("Residuals")

    # 3. Q-Q Plot
    stats.probplot(residuals, dist="norm", plot=axes[1, 0])
    axes[1, 0].set_title(f"Q-Q Plot (Validation) - {model_name}")

    # 4. Actual vs Predicted
    axes[1, 1].scatter(y_true, y_pred, alpha=0.5)
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    axes[1, 1].plot([min_val, max_val], [min_val, max_val], color='red')

    axes[1, 1].set_xlabel("Actual")
    axes[1, 1].set_ylabel("Predicted")
    axes[1, 1].set_title(f"Actual vs Predicted (Validation) - {model_name}")

    plt.tight_layout()
    plt.show()


#### 4.5.2 Classification hyperparameter optimization

##### 4.5.2.1 Classification hyperparameter distribution

In [ ]:
class_param_distributions = {
    "CatBoostClassifier": {
        "depth": [4, 6, 8],
        "learning_rate": [0.02, 0.03, 0.05, 0.1],
        "iterations": [200, 400, 600]
    },

    "RandomForestClassifier": {
        "n_estimators": [200, 400, 800, 1200],
        "max_depth": [None, 5, 10, 20],
        "max_features": ["sqrt", "log2", 0.3, 0.5]
    },

    "LogisticRegression": {
        "C": np.logspace(-3, 3, 20),
        "penalty": ["l2"],
        "solver": ["lbfgs"]
    }
}


In [ ]:
best_class_model_name = best_class_model.__class__.__name__
class_search_space = class_param_distributions[best_class_model_name]

print(f"Classification hyperparameter search space for {best_class_model_name}:")
print(class_search_space)

##### 4.5.2.2 Classificaiton hyperparameter tuning

In [ ]:
best_class_row = df_results_classification.loc[
    df_results_classification['Accuracy_valid'].idxmax()
]

In [ ]:
best_class_features = best_class_row["features"]

In [ ]:
Xtr_class_best = X_class_train[best_class_features]
Xva_class_best = X_class_valid[best_class_features]

In [ ]:
class_estimators_to_tune = {
    "CatBoostClassifier": CatBoostClassifier(
        loss_function="MultiClass",
        verbose=0,
        od_type="Iter",
        od_wait=20,
        random_state=42
    ),
    "RandomForestClassifier": RandomForestClassifier(random_state=42)
}

In [ ]:
class_hpo_results = {}

In [ ]:
for model_name, base_estimator in class_estimators_to_tune.items():
    print(f"\nRunning HPO for {model_name}...")

    search_space = class_param_distributions[model_name]

    class_search = RandomizedSearchCV(
        estimator=base_estimator,
        param_distributions=search_space,
        n_iter=10,   # increase later
        scoring="accuracy",
        cv=5,       # increase later
        n_jobs=-1,
        random_state=42,
        refit=True
    )

    class_search.fit(Xtr_class_best, y_class_train)

    best_model_hpo = class_search.best_estimator_
    best_params = class_search.best_params_

    print(f"Best hyperparameters for {model_name}:")
    print(best_params)

    class_hpo_results[model_name] = {
        "estimator": best_model_hpo,
        "params": best_params,
        "search_obj": class_search
    }


##### 4.5.2.3 Classification hyperparameter tuning diagnostics

In [ ]:
for model_name, result in class_hpo_results.items():
    print(f"\n=== Diagnostics for {model_name} ===")

    model = result["estimator"]

    preds_valid = model.predict(Xva_class_best)
    probs_valid = model.predict_proba(Xva_class_best)

    y_true = y_class_valid
    classes = sorted(y_true.unique())

    # Binarize for ROC curves
    y_valid_bin = label_binarize(y_true, classes=classes)

    # Confusion matrix
    cm = confusion_matrix(y_true, preds_valid)

    fig, axes = plt.subplots(2, 2, figsize=(16, 14))

    # ---------------------------------------------------------
    # 1. Confusion Matrix
    # ---------------------------------------------------------
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=classes,
        yticklabels=classes,
        ax=axes[0, 0]
    )
    axes[0, 0].set_title(f"Confusion Matrix (Validation) - {model_name}")
    axes[0, 0].set_xlabel("Predicted")
    axes[0, 0].set_ylabel("Actual")

    # ---------------------------------------------------------
    # 2. ROC Curves (One-vs-Rest)
    # ---------------------------------------------------------
    for i, cls in enumerate(classes):
        fpr, tpr, _ = roc_curve(y_valid_bin[:, i], probs_valid[:, i])
        roc_auc_i = auc(fpr, tpr)
        axes[0, 1].plot(fpr, tpr, label=f"Class {cls} (AUC={roc_auc_i:.3f})")

    axes[0, 1].plot([0, 1], [0, 1], "k--")
    axes[0, 1].set_title(f"ROC Curves (Validation) - {model_name}")
    axes[0, 1].set_xlabel("False Positive Rate")
    axes[0, 1].set_ylabel("True Positive Rate")
    axes[0, 1].legend()

    # ---------------------------------------------------------
    # 3. Classification Report
    # ---------------------------------------------------------
    report = classification_report(y_true, preds_valid)
    axes[1, 0].axis("off")
    axes[1, 0].text(
        0.01, 0.99,
        report,
        fontsize=12,
        family="monospace",
        va="top"
    )
    axes[1, 0].set_title(f"Classification Report (Validation) - {model_name}")

    # ---------------------------------------------------------
    # 4. Features Used
    # ---------------------------------------------------------
    axes[1, 1].axis("off")
    axes[1, 1].text(
        0.01, 0.99,
        "Features Used:\n" + "\n".join(best_class_features),
        fontsize=12,
        family="monospace",
        va="top"
    )
    axes[1, 1].set_title(f"Feature Set Used - {model_name}")

    plt.tight_layout()
    plt.show()


### 4.6 Feature selection

##### 4.5.1 Regression feature selection methods

In [ ]:
# Filter Method: SelectKBest (Regression)

def select_kbest_reg(X, y, k):
    selector = SelectKBest(score_func=f_regression, k=k)
    selector.fit(X, y)
    return X.columns[selector.get_support()].tolist()


In [ ]:
# Wrapper Method: Recursive Feature Elimination (RFE) with Linear Regression

def rfe_reg(X, y, k):
    selector = RFE(LinearRegression(), n_features_to_select=k)
    selector.fit(X, y)
    return X.columns[selector.get_support()].tolist()


In [ ]:
# Embedded Method: Lasso Regression with Cross-Validation

def lasso_reg(X, y):
    model = LassoCV(cv=5)
    model.fit(X, y)
    return X.columns[model.coef_ != 0].tolist()


##### 4.5.2 Classification feature selection methods

In [ ]:
# Filter Method: SelectKBest (Classification)

def select_kbest_class(X, y, k):
    selector = SelectKBest(score_func=f_classif, k=k)
    selector.fit(X, y)
    return X.columns[selector.get_support()].tolist()


In [ ]:
# Wrapper Method: Recursive Feature Elimination (RFE) with Logistic Regression

def rfe_class(X, y, k):
    selector = RFE(LogisticRegression(max_iter=500), n_features_to_select=k)
    selector.fit(X, y)
    return X.columns[selector.get_support()].tolist()

In [ ]:
# Embedded Method: Lasso Regression with Cross-Validation for Classification (using Logistic Regression)

def lasso_class(X, y):
    model = LassoCV(cv=5)
    model.fit(X, y)
    return X.columns[model.coef_ != 0].tolist()

#### 4.5.3 Optimized regression model

In [ ]:
ytr_reg = y_reg_train

In [ ]:
print("\n=== Running Regression Feature Selection on Best Feature Set ===")

# 1. KBest (k=20 or whatever you choose)
selected_kbest_reg = select_kbest_reg(Xtr_reg_best, ytr_reg, k=20)
print(f"KBest selected {len(selected_kbest_reg)} features.")

# 2. RFE (k=20 or whatever you choose)
selected_rfe_reg = rfe_reg(Xtr_reg_best, ytr_reg, k=20)
print(f"RFE selected {len(selected_rfe_reg)} features.")

# 3. Lasso (automatic number of features)
selected_lasso_reg = lasso_reg(Xtr_reg_best, ytr_reg)
print(f"Lasso selected {len(selected_lasso_reg)} features.")


In [ ]:
reg_feature_sets = {
    "kbest": selected_kbest_reg,
    "rfe": selected_rfe_reg,
    "lasso": selected_lasso_reg
}


In [ ]:
for name, feats in reg_feature_sets.items():
    print(f"\n{name.upper()} Regression Features ({len(feats)}):")
    for f in feats:
        print(" -", f)


In [ ]:
reg_hpo_results["CatBoostRegressor"]["estimator"]
reg_hpo_results["RandomForestRegressor"]["estimator"]

In [ ]:
X_reg_train, y_reg_train
X_reg_valid, y_reg_valid

In [ ]:
regression_final_results = []

In [ ]:
print("\n=== Building and Evaluating 6 Regression Models ===")

for model_name, hpo_info in reg_hpo_results.items():
    tuned_model = hpo_info["estimator"]
    tuned_params = hpo_info["params"]

    for fs_name, fs_features in reg_feature_sets.items():
        print(f"\nTraining {model_name} with {fs_name.upper()} features...")

        # Extract feature subset
        Xtr_fs = X_reg_train[fs_features]
        Xva_fs = X_reg_valid[fs_features]

        # Clone the tuned model with frozen hyperparameters
        ModelClass = tuned_model.__class__
        if model_name == "CatBoostRegressor":
            model = ModelClass(**tuned_params, verbose=False)
        else:
            model = ModelClass(**tuned_params)


        # Fit
        model.fit(Xtr_fs, y_reg_train)

        # Predict
        preds = model.predict(Xva_fs)

        y_true = y_reg_valid
        y_pred = preds
        
        # Compute metrics
        mse = mean_squared_error(y_true, y_pred)
        rmse = root_mean_squared_error(y_true, y_pred)
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)

        print(f"RMSE: {rmse:.4f}, MSE: {mse:.4f}, MAE: {mae:.4f}, R²: {r2:.4f}")

        # Store results
        regression_final_results.append({
            "model": model_name,
            "feature_set": fs_name,
            "n_features": len(fs_features),
            "rmse": rmse,
            "mse": mse,
            "mae": mae,
            "r2": r2,
            "features": fs_features,
            "params": tuned_params
        })


In [ ]:
df_regression_final = pd.DataFrame(regression_final_results)
df_regression_final.sort_values("rmse", inplace=True)
df_regression_final

#### 4.5.4 Optimized classification model

In [ ]:
ytr_class = y_class_train

In [ ]:
print("\n=== Running Classification Feature Selection on Best Feature Set ===")

# Extract X and y for the best classification feature set
Xtr_class_best = X_class_train[best_class_features]
ytr_class = y_class_train

# 1. KBest
selected_kbest_class = select_kbest_class(Xtr_class_best, ytr_class, k=20)
print(f"KBest selected {len(selected_kbest_class)} features.")

# 2. RFE
selected_rfe_class = rfe_class(Xtr_class_best, ytr_class, k=20)
print(f"RFE selected {len(selected_rfe_class)} features.")

# 3. Lasso
selected_lasso_class = lasso_class(Xtr_class_best, ytr_class)
print(f"Lasso selected {len(selected_lasso_class)} features.")

In [ ]:
class_feature_sets = {
    "kbest": selected_kbest_class,
    "rfe": selected_rfe_class,
    "lasso": selected_lasso_class
}

In [ ]:
for name, feats in class_feature_sets.items():
    print(f"\n{name.upper()} Classification Features ({len(feats)}):")
    for f in feats:
        print(" -", f)

In [ ]:
class_hpo_results["CatBoostClassifier"]["estimator"]
class_hpo_results["RandomForestClassifier"]["estimator"]

In [ ]:
classification_final_results = []


In [ ]:
print("\n=== Building and Evaluating 6 Classification Models ===")

for model_name, hpo_info in class_hpo_results.items():
    tuned_model = hpo_info["estimator"]
    tuned_params = hpo_info["params"]

    for fs_name, fs_features in class_feature_sets.items():
        print(f"\nTraining {model_name} with {fs_name.upper()} features...")

        # Extract feature subset
        Xtr_fs = X_class_train[fs_features]
        Xva_fs = X_class_valid[fs_features]

        # Clone tuned model with frozen hyperparameters
        ModelClass = tuned_model.__class__
        model = ModelClass(**tuned_params)

        # Fit
        model.fit(Xtr_fs, y_class_train)

        # Predict
        preds = model.predict(Xva_fs)
        probs = model.predict_proba(Xva_fs)

        # Metrics
        acc = accuracy_score(y_class_valid, preds)
        f1 = f1_score(y_class_valid, preds, average="weighted")
        precision = precision_score(y_class_valid, preds, average="weighted", zero_division=0)
        recall = recall_score(y_class_valid, preds, average="weighted", zero_division=0)

        print(f"ACC: {acc:.4f}, F1: {f1:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}")

        # Store results
        classification_final_results.append({
            "model": model_name,
            "feature_set": fs_name,
            "n_features": len(fs_features),
            "accuracy": acc,
            "f1": f1,
            "precision": precision,
            "recall": recall,
            "features": fs_features,
            "params": tuned_params
        })


In [ ]:
df_classification_final = pd.DataFrame(classification_final_results)
df_classification_final.sort_values("accuracy", ascending=False, inplace=True)
df_classification_final

## 5.0 Evaluation

### 5.1 Regression evaluation

In [ ]:
X_reg_train_full = pd.concat([X_reg_train, X_reg_valid])
y_reg_train_full = pd.concat([y_reg_train, y_reg_valid])

In [ ]:
regression_test_results = []

In [ ]:
print("\n=== Regression Test-Set Evaluation ===")

for _, row in df_regression_final.iterrows():
    model_name = row["model"]
    feats = row["features"]
    params = row["params"]

    # Rebuild model from stored hyperparameters
    if model_name == "CatBoostRegressor":
        ModelClass = CatBoostRegressor
    elif model_name == "RandomForestRegressor":
        ModelClass = RandomForestRegressor
    else:
        raise ValueError(f"Unknown model: {model_name}")

    if model_name == "CatBoostRegressor":
        model = ModelClass(**params, verbose=False)
    else:
        model = ModelClass(**params)


    # Train on full training data
    Xtr = X_reg_train_full[feats]   # train + valid combined
    ytr = y_reg_train_full
    Xte = X_reg_test[feats]

    model.fit(Xtr, ytr)

    # Predict on test set
    preds = model.predict(Xte)

    # Metrics
    mse = mean_squared_error(y_reg_test, preds)
    rmse = root_mean_squared_error(y_reg_test, preds)
    mae = mean_absolute_error(y_reg_test, preds)
    r2 = r2_score(y_reg_test, preds)

    print(f"{model_name} ({row['feature_set']}): RMSE={rmse:.4f}, MAE={mae:.4f}, R²={r2:.4f}")

    regression_test_results.append({
        "model": model_name,
        "feature_set": row["feature_set"],
        "rmse": rmse,
        "mse": mse,
        "mae": mae,
        "r2": r2,
        "n_features": len(feats),
        "features": feats,
        "params": params
    })


### 5.2 Classification evaluaton

In [ ]:
X_class_train_full = pd.concat([X_class_train, X_class_valid])
y_class_train_full = pd.concat([y_class_train, y_class_valid])

In [ ]:
classification_test_results = []

In [ ]:
print("\n=== Classification Test-Set Evaluation ===")

for _, row in df_classification_final.iterrows():
    model_name = row["model"]
    feats = row["features"]
    params = row["params"]

    # Rebuild model from stored hyperparameters
    if model_name == "CatBoostClassifier":
        model = CatBoostClassifier(**params, verbose=False)
    elif model_name == "RandomForestClassifier":
        model = RandomForestClassifier(**params)
    else:
        raise ValueError(f"Unknown model: {model_name}")

    # Train on full training data
    Xtr = X_class_train_full[feats]
    ytr = y_class_train_full
    Xte = X_class_test[feats]

    model.fit(Xtr, ytr)

    # Predict on test set
    preds = model.predict(Xte)

    # Metrics
    acc = accuracy_score(y_class_test, preds)
    precision = precision_score(y_class_test, preds, average="weighted", zero_division=0)
    recall = recall_score(y_class_test, preds, average="weighted", zero_division=0)
    f1 = f1_score(y_class_test, preds, average="weighted")

    print(
        f"{model_name} ({row['feature_set']}): "
        f"ACC={acc:.4f}, Precision={precision:.4f}, "
        f"Recall={recall:.4f}, F1={f1:.4f}"
    )

    classification_test_results.append({
        "model": model_name,
        "feature_set": row["feature_set"],
        "n_features": len(feats),
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "features": feats,
        "params": params
    })


### 5.3 Regression diagnostics

#### 5.3.1 Regression statistics

##### 5.3.1.1 Residual mean and standard deviation

In [ ]:
residual_stats = []

In [ ]:
for _, row in df_regression_final.iterrows():
    model_name = row["model"]
    feats = row["features"]
    params = row["params"]

    # Rebuild model
    if model_name == "CatBoostRegressor":
        model = CatBoostRegressor(**params, verbose=False)
    else:
        model = RandomForestRegressor(**params)

    # Fit on full training data
    model.fit(X_reg_train_full[feats], y_reg_train_full)

    # Predict on test data
    y_pred = model.predict(X_reg_test[feats])
    y_true = y_reg_test

    # Residuals
    residuals = y_true - y_pred

    residual_stats.append({
        "label": f"{model_name} ({row['feature_set']})",
        "Residual_Mean": np.mean(residuals),
        "Residual_Std": np.std(residuals),
        "Residuals": residuals
    })

In [ ]:
df_residual_stats = pd.DataFrame(residual_stats)
df_residual_stats[["label", "Residual_Mean", "Residual_Std"]]

##### 5.3.1.2 Residual distribution

In [ ]:
diagnostic_data = []

In [ ]:
for _, row in df_regression_final.iterrows():
    model_name = row["model"]
    feats = row["features"]
    params = row["params"]

    # Rebuild model
    if model_name == "CatBoostRegressor":
        model = CatBoostRegressor(**params, verbose=False)
    else:
        model = RandomForestRegressor(**params)

    # Fit on full training data
    model.fit(X_reg_train_full[feats], y_reg_train_full)

    # Predict on test data
    y_pred_log = model.predict(X_reg_test[feats])
    y_true_log = y_reg_test

    residuals = y_true_log - y_pred_log

    diagnostic_data.append({
        "label": f"{model_name} ({row['feature_set']})",
        "y_true": y_true_log,
        "y_pred": y_pred_log,
        "residuals": residuals
    })

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes = axes.flatten()

for ax, data in zip(axes, diagnostic_data):
    ax.hist(data["residuals"], bins=50, color="steelblue", alpha=0.7)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_title(f"Residual Distribution\n{data['label']}")

plt.tight_layout()
plt.show()


##### 5.3.1.3 Residual vs prediction

In [ ]:
fig, axes = plt.subplots(2,3, figsize=(18, 8))
axes = axes.flatten()

for ax, data in zip(axes, diagnostic_data):
    ax.scatter(data["y_pred"], data["residuals"], alpha=0.5, s=12)
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title(f"Residuals vs Predicted\n{data['label']}")
    ax.set_xlabel("Predicted (log)")
    ax.set_ylabel("Residuals")

plt.tight_layout()
plt.show()


##### 5.3.1.4 Q-Q plot

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes = axes.flatten()

for ax, data in zip(axes, diagnostic_data):
    stats.probplot(data["residuals"], dist="norm", plot=ax)
    ax.set_title(f"Q–Q Plot\n{data['label']}")

plt.tight_layout()
plt.show()


##### 5.3.1.5 Actual vs predicted

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes = axes.flatten()

for ax, data in zip(axes, diagnostic_data):
    ax.scatter(data["y_true"], data["y_pred"], alpha=0.5, s=12)

    min_val = min(data["y_true"].min(), data["y_pred"].min())
    max_val = max(data["y_true"].max(), data["y_pred"].max())
    ax.plot([min_val, max_val], [min_val, max_val], color="red")

    ax.set_title(f"Actual vs Predicted\n{data['label']}")
    ax.set_xlabel("Actual (log)")
    ax.set_ylabel("Predicted (log)")

plt.tight_layout()
plt.show()


In [ ]:
feature_panels_reg = []

In [ ]:
for _, row in df_regression_final.iterrows():
    label = f"{row['model']} ({row['feature_set']})"
    feats = row["features"]

    feature_panels_reg.append({
        "label": label,
        "features": feats
    })

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes = axes.flatten()

for ax, item in zip(axes, feature_panels_reg):
    ax.axis("off")
    feat_text = "\n".join(item["features"])
    ax.text(
        0.0, 0.5,
        f"Features Used — {item['label']}:\n\n{feat_text}",
        fontsize=10,
        va="center",
        ha="left",
        family="monospace"
    )

# Turn off unused axes (if fewer than 6 models)
for ax in axes[len(feature_panels_reg):]:
    ax.axis("off")

plt.tight_layout()
plt.show()


##### 5.3.1.6 Error by log price quantile

In [ ]:
quantiles = pd.qcut(y_reg_test, 5, duplicates="drop")

In [ ]:
quantile_error_rows = []

In [ ]:
for _, row in df_regression_final.iterrows():
    model_name = row["model"]
    feats = row["features"]
    params = row["params"]

    # Rebuild model
    if model_name == "CatBoostRegressor":
        model = CatBoostRegressor(**params, verbose=False)
    else:
        model = RandomForestRegressor(**params)

    # Fit on full training data
    model.fit(X_reg_train_full[feats], y_reg_train_full)

    # Predict on test data
    y_pred_log = model.predict(X_reg_test[feats])
    y_true_log = y_reg_test

    abs_error = np.abs(y_true_log - y_pred_log)

    df_temp = pd.DataFrame({
        "label": f"{model_name} ({row['feature_set']})",
        "actual": y_true_log,
        "pred": y_pred_log,
        "abs_error": abs_error,
        "quantile": quantiles
    })

    quantile_means = df_temp.groupby("quantile")["abs_error"].mean()

    for q, val in quantile_means.items():
        quantile_error_rows.append({
            "label": f"{model_name} ({row['feature_set']})",
            "quantile": str(q),
            "mean_abs_error": val
        })


In [ ]:
df_quantile_errors = pd.DataFrame(quantile_error_rows)

In [ ]:
df_quantile_errors.pivot(
    index="quantile",
    columns="label",
    values="mean_abs_error"
)

#### 5.3.2 Regression spider diagram

In [ ]:
metric_names = ["MAE", "MSE", "RMSE", "R2"]

In [ ]:
model_metrics = {}

In [ ]:
for _, row in df_regression_final.iterrows():
    model_name = row["model"]
    feats = row["features"]
    params = row["params"]

    # Rebuild model
    if model_name == "CatBoostRegressor":
        model = CatBoostRegressor(**params, verbose=False)
    else:
        model = RandomForestRegressor(**params)

    # Fit on full training data
    model.fit(X_reg_train_full[feats], y_reg_train_full)

    # Predict on test data
    y_pred = model.predict(X_reg_test[feats])
    y_true = y_reg_test

    # Store metrics
    label = f"{model_name} ({row['feature_set']})"
    model_metrics[label] = {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mean_squared_error(y_true, y_pred),
        "RMSE": root_mean_squared_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }

In [ ]:
df_raw = pd.DataFrame(model_metrics).T
df_raw

In [ ]:
def spider_multi(ax, metrics, df, title):
    N = len(metrics)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    for label in df.index:
        vals = df.loc[label, metrics].tolist()
        vals += vals[:1]
        ax.plot(angles, vals, linewidth=2, label=label)
        ax.fill(angles, vals, alpha=0.10)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics)
    ax.set_title(title, fontsize=14)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

spider_multi(
    ax=ax,
    metrics=metric_names,
    df=df_raw,
    title="Regression Model Comparison"
)

plt.tight_layout()
plt.show()

#### 5.3.3 Test original best regression model

In [ ]:
best_reg_params = best_reg_model.get_params()

In [ ]:
catboost_allowed = {
    "iterations", "learning_rate", "depth", "l2_leaf_reg",
    "loss_function", "random_seed", "subsample", "colsample_bylevel",
    "min_data_in_leaf", "max_depth", "grow_policy"
}

rf_allowed = {
    "n_estimators", "max_depth", "min_samples_split",
    "min_samples_leaf", "max_features", "bootstrap",
    "random_state"
}

In [ ]:
if best_reg_model_name == "CatBoostRegressor":
    filtered_params = {k: v for k, v in best_reg_params.items() if k in catboost_allowed}
    final_reg_model = CatBoostRegressor(**filtered_params, verbose=False)

else:
    filtered_params = {k: v for k, v in best_reg_params.items() if k in rf_allowed}
    final_reg_model = RandomForestRegressor(**filtered_params)

In [ ]:
# Fit on full training data
final_reg_model.fit(
    X_reg_train_full[best_reg_features],
    y_reg_train_full
)

In [ ]:
y_pred_log = final_reg_model.predict(X_reg_test[best_reg_features])

In [ ]:
print(best_reg_model)
print(best_reg_features)
print(best_reg_model_name)
print(final_reg_model)

In [ ]:
y_true_log = y_reg_test
residuals = y_true_log - y_pred_log

In [ ]:
original_reg_metrics = {
    "MAE": mean_absolute_error(y_true_log, y_pred_log),
    "MSE": mean_squared_error(y_true_log, y_pred_log),
    "RMSE": root_mean_squared_error(y_true_log, y_pred_log),
    "R2": r2_score(y_true_log, y_pred_log)
}

In [ ]:
pd.DataFrame([original_reg_metrics], index=["Original Best Regression Model"])

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Residual Distribution
axes[0, 0].hist(residuals, bins=50, color='steelblue', edgecolor='black')
axes[0, 0].set_title("Residual Distribution (Test)")
axes[0, 0].set_xlabel("Residual")
axes[0, 0].set_ylabel("Frequency")

# 2. Residuals vs Predicted
axes[0, 1].scatter(y_pred_log, residuals, alpha=0.5)
axes[0, 1].axhline(0, color='red', linestyle='--')
axes[0, 1].set_title("Residuals vs Predicted (Test)")
axes[0, 1].set_xlabel("Predicted (log)")
axes[0, 1].set_ylabel("Residuals")

# 3. Q-Q Plot
stats.probplot(residuals, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title("Q-Q Plot (Test)")

# 4. Actual vs Predicted
axes[1, 1].scatter(y_true_log, y_pred_log, alpha=0.5)
min_val = min(y_true_log.min(), y_pred_log.min())
max_val = max(y_true_log.max(), y_pred_log.max())
axes[1, 1].plot([min_val, max_val], [min_val, max_val], color='red')

axes[1, 1].set_xlabel("Actual (log)")
axes[1, 1].set_ylabel("Predicted (log)")
axes[1, 1].set_title("Actual vs Predicted (Test)")

plt.tight_layout()
plt.show()


### 5.4 Classification diagnostics

#### 5.4.1 Classification statistics

##### 5.4.1.1 Confusion matrix

In [ ]:
class_labels = sorted(y_class_test.unique())

In [ ]:
confusion_data = []

In [ ]:
for _, row in df_classification_final.iterrows():
    model_name = row["model"]
    feats = row["features"]
    params = row["params"]

    # Rebuild model
    if model_name == "CatBoostClassifier":
        model = CatBoostClassifier(**params, verbose=False)
    else:
        model = RandomForestClassifier(**params)

    # Fit on full training data
    model.fit(X_class_train_full[feats], y_class_train_full)

    # Predict on test data
    preds = model.predict(X_class_test[feats])

    # Confusion matrix
    cm = confusion_matrix(y_class_test, preds)

    confusion_data.append({
        "label": f"{model_name} ({row['feature_set']})",
        "cm": cm
    })


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for ax, item in zip(axes, confusion_data):
    sns.heatmap(
        item["cm"],
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_labels,
        yticklabels=class_labels,
        ax=ax
    )
    ax.set_title(f"Confusion Matrix — {item['label']}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()



##### 5.4.1.2 Model features

In [ ]:
feature_panels = []

for _, row in df_classification_final.iterrows():
    label = f"{row['model']} ({row['feature_set']})"
    feats = row["features"]

    feature_panels.append({
        "label": label,
        "features": feats
    })

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes = axes.flatten()

for ax, item in zip(axes, feature_panels):
    ax.axis("off")
    feat_text = "\n".join(item["features"])
    ax.text(
        0.0, 0.5,
        f"Features Used — {item['label']}:\n\n{feat_text}",
        fontsize=10,
        va="center",
        ha="left",
        family="monospace"
    )

plt.tight_layout()
plt.show()


##### 5.4.1.3 Classification report

In [ ]:
report_data = []

In [ ]:
for _, row in df_classification_final.iterrows():
    model_name = row["model"]
    feats = row["features"]
    params = row["params"]

    # Rebuild model
    if model_name == "CatBoostClassifier":
        model = CatBoostClassifier(**params, verbose=False)
    else:
        model = RandomForestClassifier(**params)

    # Fit on full training data
    model.fit(X_class_train_full[feats], y_class_train_full)

    # Predict on test data
    preds = model.predict(X_class_test[feats])

    # Generate report text
    report_text = classification_report(y_class_test, preds)

    report_data.append({
        "label": f"{model_name} ({row['feature_set']})",
        "report": report_text
    })


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for ax, item in zip(axes, report_data):
    ax.axis("off")
    ax.text(
        0.0, 1.0,
        f"Classification Report — {item['label']}\n\n{item['report']}",
        fontsize=10,
        va="top",
        ha="left",
        family="monospace"
    )

# Turn off any unused axes (should be none for 6 models)
for ax in axes[len(report_data):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

##### 5.4.1.4 ROC-AUC

In [ ]:
classes = sorted(y_class_test.unique())
y_test_bin = label_binarize(y_class_test, classes=classes)

In [ ]:
roc_data = []

In [ ]:
for _, row in df_classification_final.iterrows():
    model_name = row["model"]
    feats = row["features"]
    params = row["params"]

    # Rebuild model
    if model_name == "CatBoostClassifier":
        model = CatBoostClassifier(**params, verbose=False)
    else:
        model = RandomForestClassifier(**params)

    # Fit on full training data
    model.fit(X_class_train_full[feats], y_class_train_full)

    # Predict probabilities
    probs = model.predict_proba(X_class_test[feats])

    # Align probability columns with test-set classes
    class_indices = [model.classes_.tolist().index(c) for c in classes]
    probs_filtered = probs[:, class_indices]

    # Normalize rows
    probs_norm = probs_filtered / probs_filtered.sum(axis=1, keepdims=True)

    # Weighted OVR ROC-AUC
    roc_auc_weighted = roc_auc_score(
        y_class_test,
        probs_norm,
        multi_class="ovr",
        average="weighted"
    )

    roc_data.append({
        "label": f"{model_name} ({row['feature_set']})",
        "probs": probs_norm,
        "roc_auc_weighted": roc_auc_weighted
    })


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(26, 12))
axes = axes.flatten()

for ax, item in zip(axes, roc_data):
    probs = item["probs"]

    for i, cls in enumerate(classes):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], probs[:, i])
        roc_auc_i = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f"Class {cls} (AUC={roc_auc_i:.3f})")

    ax.plot([0,1], [0,1], "k--")
    ax.set_title(
        f"ROC Curves — {item['label']}\nWeighted AUC={item['roc_auc_weighted']:.3f}",
        fontsize=12
    )
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend(fontsize=8)

# Turn off unused axes (should be none for 6 models)
for ax in axes[len(roc_data):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

#### 5.4.2 Classification spider diagram

In [ ]:
metric_names = ["Accuracy", "Precision", "Recall", "F1"]

In [ ]:
class_metrics = {}

In [ ]:
for _, row in df_classification_final.iterrows():
    model_name = row["model"]
    feats = row["features"]
    params = row["params"]

    # Rebuild model
    if model_name == "CatBoostClassifier":
        model = CatBoostClassifier(**params, verbose=False)
    else:
        model = RandomForestClassifier(**params)

    # Fit on full training data
    model.fit(X_class_train_full[feats], y_class_train_full)

    # Predict on test data
    y_pred = model.predict(X_class_test[feats])
    y_true = y_class_test

    # Store metrics
    label = f"{model_name} ({row['feature_set']})"
    class_metrics[label] = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "F1": f1_score(y_true, y_pred, average="weighted", zero_division=0)
    }


In [ ]:
df_class_raw = pd.DataFrame(class_metrics).T
df_class_raw

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

spider_multi(
    ax=ax,
    metrics=metric_names,
    df=df_class_raw,
    title="Classification Model Comparison (Test Set)"
)

plt.tight_layout()
plt.show()


#### 5.4.3 Test original best classification model

In [ ]:
best_class_params = best_class_model.get_params()

In [ ]:
# Split params by model type
catboost_allowed = {
    "iterations", "learning_rate", "depth", "l2_leaf_reg",
    "loss_function", "random_seed", "subsample", "colsample_bylevel",
    "min_data_in_leaf", "max_depth", "grow_policy"
}

rf_allowed = {
    "n_estimators", "max_depth", "min_samples_split",
    "min_samples_leaf", "max_features", "bootstrap",
    "random_state"
}

In [ ]:
if best_class_model_name == "CatBoostClassifier":
    filtered_params = {k: v for k, v in best_class_params.items() if k in catboost_allowed}
    final_class_model = CatBoostClassifier(**filtered_params, verbose=False)

else:
    filtered_params = {k: v for k, v in best_class_params.items() if k in rf_allowed}
    final_class_model = RandomForestClassifier(**filtered_params)


In [ ]:
final_class_model.fit(
    X_class_train_full[best_class_features],
    y_class_train_full
)

In [ ]:
preds_test = final_class_model.predict(X_class_test[best_class_features])
probs_test = final_class_model.predict_proba(X_class_test[best_class_features])

In [ ]:
y_true = y_class_test
classes = sorted(y_true.unique())
y_test_bin = label_binarize(y_true, classes=classes)

In [ ]:
cm = confusion_matrix(y_true, preds_test)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# 1. Confusion Matrix
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=classes,
    yticklabels=classes,
    ax=axes[0, 0]
)
axes[0, 0].set_title("Confusion Matrix (Test)")
axes[0, 0].set_xlabel("Predicted")
axes[0, 0].set_ylabel("Actual")

# 2. ROC Curves
for i, cls in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], probs_test[:, i])
    roc_auc_i = auc(fpr, tpr)
    axes[0, 1].plot(fpr, tpr, label=f"Class {cls} (AUC={roc_auc_i:.3f})")

axes[0, 1].plot([0, 1], [0, 1], "k--")
axes[0, 1].set_title("ROC Curves (Test)")
axes[0, 1].set_xlabel("False Positive Rate")
axes[0, 1].set_ylabel("True Positive Rate")
axes[0, 1].legend()

# 3. Classification Report
report = classification_report(y_true, preds_test)
axes[1, 0].axis("off")
axes[1, 0].text(
    0.01, 0.99,
    report,
    fontsize=12,
    family="monospace",
    va="top"
)
axes[1, 0].set_title("Classification Report (Test)")

# 4. Features Used
axes[1, 1].axis("off")
axes[1, 1].text(
    0.01, 0.99,
    "Features Used:\n" + "\n".join(best_class_features),
    fontsize=12,
    family="monospace",
    va="top"
)
axes[1, 1].set_title("Feature Set Used")

plt.tight_layout()
plt.show()


In [ ]:
print("\n=== Original Best Classification Model: Test-Set Evaluation ===")

# Rebuild the original best model
if best_class_model_name == "CatBoostClassifier":
    original_class_model = CatBoostClassifier(**best_class_params, verbose=False)
elif best_class_model_name == "RandomForestClassifier":
    original_class_model = RandomForestClassifier(**best_class_params)
else:
    raise ValueError(f"Unknown model: {best_class_model_name}")

# Fit on full training data
Xtr = X_class_train_full[best_class_features]
ytr = y_class_train_full
Xte = X_class_test[best_class_features]

original_class_model.fit(Xtr, ytr)

# Predict
preds = original_class_model.predict(Xte)

# Metrics
acc = accuracy_score(y_class_test, preds)
precision = precision_score(y_class_test, preds, average="weighted", zero_division=0)
recall = recall_score(y_class_test, preds, average="weighted", zero_division=0)
f1 = f1_score(y_class_test, preds, average="weighted")

original_class_metrics = {
    "Accuracy": acc,
    "Precision": precision,
    "Recall": recall,
    "F1": f1
}

pd.DataFrame([original_class_metrics], index=["Original Best Classification Model"])
